# BFT65B – FDIC early-warning pipeline: executed run
Student: Lã Thị Thu Trang (11231015). Every cell calls the same code as `python run.py <stage>`. Data must first be downloaded with `python run.py ingest` (resumable, ~11 min).

In [1]:
import os, sys, json, subprocess
os.chdir('..') if os.path.basename(os.getcwd()) == 'notebooks' else None
sys.path.insert(0, 'src')
import polars as pl
pl.Config.set_tbl_rows(30); pl.Config.set_tbl_cols(12)
def run(stage):
    out = subprocess.run([sys.executable, 'run.py', stage], capture_output=True, text=True)
    print(out.stdout[-2500:]); print(out.stderr[-500:] if out.returncode else '')

## Part 1 – completeness and curated storage

In [2]:
run('verify')

=== verify
{
 "quarters_expected": 170,
 "quarters_downloaded": 170,
 "quarters_missing": [],
 "quarters_incomplete": [],
 "checksum_mismatch": [],
 "financials_rows": 1682738,
 "api_total_sum": 1682738,
 "distinct_banks": 24064,
 "duplicate_cert_repdte": 0,
 "failures_rows": 4117,
 "institutions_rows": 27838,
 "download_first": "2026-11-20T18:40:23Z",
 "download_last": "2026-11-20T18:51:02Z",
 "complete": true
}
=== verify done in 16.7s




In [3]:
run('store')
pl.read_csv('reports/tables/storage_benchmark.csv').select('format','query','size_mb','median_s','speedup_vs_csv','peak_rss_mb')

=== store
curated financials: 1,682,738 rows in 43 year partitions
=== store done in 3.5s




format,query,size_mb,median_s,speedup_vs_csv,peak_rss_mb
str,str,f64,f64,f64,f64
"""csv""","""full_load""",363.63555,0.725486,1.0,964.503906
"""csv""","""agg_2cols""",363.63555,0.298515,1.0,460.910156
"""csv""","""filter_1yr_state""",363.63555,0.351032,1.0,470.273438
"""csv_gzip""","""full_load""",163.311066,1.275725,0.568685,758.296875
"""csv_gzip""","""agg_2cols""",163.311066,1.325993,0.225126,722.296875
"""csv_gzip""","""filter_1yr_state""",163.311066,1.295593,0.270943,725.160156
"""parquet_none""","""full_load""",275.371934,0.123042,5.896226,574.265625
"""parquet_none""","""agg_2cols""",275.371934,0.020598,14.492517,116.40625
"""parquet_none""","""filter_1yr_state""",275.371934,0.006521,53.829031,94.53125


## Part 2 – point-in-time panel and data-quality report

In [4]:
run('panel')
pl.read_csv('reports/tables/panel_checks.csv')

=== panel
{
 "failures": {
  "rows_raw": 4117,
  "restype_counts": {
   "FAILURE": 3524,
   "ASSISTANCE": 593
  },
  "missing_cert": 488,
  "duplicate_cert": 6,
  "rows_used": 3036,
  "used_since_1984": 2845
 },
 "rows_financials": 1682738,
 "rows_removed_on_or_after_failure": 1211,
 "rows_panel": 1681527,
 "positive_rows": 10904,
 "failed_banks_with_positive_row": 2839,
 "failed_banks_in_panel": 2839,
 "positives_failing_before_available": 1492,
 "labels_asof": "2026-11-20",
 "exit_banks": {
  "other_exit": 258,
  "merger_or_absorption": 16766,
  "active": 4201,
  "failure": 2839
 },
 "failed_before_first_report": 2
}
panel checks: 7 passed
=== panel done in 1.6s




check,passed,detail
str,bool,str
"""(CERT, REPDTE) unique""",true,""""""
"""every positive fails strictly …",true,""""""
"""every positive fails within 4 …",true,"""{1: 2528, 2: 2812, 3: 2793, 4:…"
"""no negative row has a failure …",true,"""0"""
"""no row on/after its failure da…",true,""""""
"""no positive without a failure …",true,""""""
"""at most 4 positive quarters pe…",true,"""mean 3.84"""


In [5]:
run('quality')
print(open('reports/data_quality_report.md').read()[:6000])

--|---|---|---|---|---|---|
| EQ | 0.25 | 0 | 43 | None | scattered | 0.00 | 0.25 |
| BRO | 1.00 | 0 | 43 | None | scattered | 0.53 | 1.00 |
| LNRECONS | 0.17 | 0 | 25 | None | scattered | 0.00 | 0.17 |
| LNCON | 0.23 | 0 | 42 | None | scattered | 0.00 | 0.24 |
| LNATRES | 0.25 | 0 | 43 | None | scattered | 0.00 | 0.25 |
| NCLNLS | 0.15 | 0 | 3 | None | scattered | 0.00 | 0.15 |
| P3ASSET | 57.77 | 17 | 0 | 2,000 | structural (not collected in some years) | 79.15 | 57.60 |
| P9ASSET | 4.62 | 0 | 7 | None | scattered | 21.53 | 4.46 |
| NAASSET | 4.62 | 0 | 6 | None | scattered | 21.53 | 4.46 |
| NTLNLS | 4.75 | 0 | 43 | None | scattered | 21.53 | 4.59 |
| ORE | 0.25 | 0 | 43 | None | scattered | 0.00 | 0.25 |
| NETINC | 0.28 | 0 | 43 | None | scattered | 0.00 | 0.28 |
| INTINC | 0.28 | 0 | 43 | None | scattered | 0.00 | 0.28 |
| EINTEXP | 0.38 | 0 | 43 | None | scattered | 0.00 | 0.38 |
| NONII | 0.28 | 0 | 43 | None | scattered | 0.00 | 0.28 |
| NONIX | 0.28 | 0 | 43 | None | scattered

## Part 3/4 – features (Polars lazy), look-ahead test, engines, PySpark parity

In [6]:
run('features')
print(json.load(open('reports/tables/features_lookahead_test.json')))
print(open('reports/tables/plan_bench_polars_optimized.txt').read()[:3000])

=== features
features: 1,681,527 rows x 40 features; look-ahead test: {'cutoff': '2001-12-31', 'rows_compared': 1010188, 'features': 40, 'features_changed_by_future_rows': [], 'passed': True}
=== features done in 4.9s


{'cutoff': '2001-12-31', 'rows_compared': 1010188, 'features': 40, 'features_changed_by_future_rows': [], 'passed': True}
simple π 11/11 ["CERT", "REPDTE", "y", ... 8 other columns]
   WITH_COLUMNS:
   [when((col("_qi") - col("_qi").shift([dyn int: 4]).over([col("CERT")])) == 4).then(col("npl_ratio") - col("npl_ratio").shift([dyn int: 4]).over([col("CERT")])).otherwise(null.cast(Float64)).alias("d4_npl_ratio"), col("roa_q").rolling_mean().over([col("CERT")]).alias("roa_mean4"), (col("npl_ratio").rank().over([col("REPDTE")]) / col("npl_ratio").count().over([col("REPDTE")]).cast(Float64)).alias("npl_rank_q")] 
    simple π 9/9 ["CERT", "REPDTE", "_qi", "y", ... 5 other columns]
      simple π 18/18 ["CERT", "REPDTE", "ASSET", ... 15 other columns]
         WITH_COLUMNS:
 

In [7]:
pl.read_csv('reports/tables/processing_benchmark.csv')

scale,engine,rows,median_s,min_s,max_s,spark_startup_s,peak_mb,count_mismatches,max_rel_diff_sums,speedup_vs_pandas
i64,str,i64,f64,f64,f64,f64,f64,i64,f64,f64
1,"""pandas""",1682738,2.580238,2.573602,2.635971,0.0,1280.160156,0,2.0960e-16,1.0
1,"""polars_eager""",1682738,1.199939,1.1602,1.216543,0.0,1102.730469,0,0.0,2.150308
1,"""polars_lazy""",1682738,1.082529,0.980437,1.128028,0.0,630.300781,0,0.0,2.383527
1,"""polars_streaming""",1682738,0.793076,0.722216,0.954699,0.0,644.625,0,4.8732e-16,3.253454
1,"""pyspark""",1682738,20.736216,20.627247,21.555544,7.577436,1981.773438,0,3.5037e-14,0.124431
3,"""pandas""",5048214,7.615765,7.530021,7.721381,0.0,2189.628906,0,2.2109e-16,1.0
3,"""polars_eager""",5048214,3.174447,2.875883,3.272182,0.0,1758.148438,0,0.0,2.399084
3,"""polars_lazy""",5048214,2.83071,2.789777,2.855626,0.0,1389.871094,0,0.0,2.690408
3,"""polars_streaming""",5048214,1.896546,1.883697,1.904057,0.0,1407.050781,0,3.1120e-16,4.015597


In [8]:
run('spark')
print(open('reports/tables/plan_spark_formatted.txt').read()[:2500])

=== spark
{
 "rows_polars": 1682738,
 "rows_spark": 1682738,
 "rows_only_in_one_engine": 0,
 "label_mismatches": 0,
 "spark_session_start_s": 25.364300728000103,
 "spark_compute_write_s": 11.030684251000025,
 "polars_compute_s": 0.5882285499999398,
 "columns": [
  {
   "column": "equity_to_assets",
   "null_disagreements": 0,
   "max_abs_diff": 0.0
  },
  {
   "column": "npl_ratio",
   "null_disagreements": 0,
   "max_abs_diff": 0.0
  },
  {
   "column": "roa_q",
   "null_disagreements": 0,
   "max_abs_diff": 0.0
  },
  {
   "column": "nim_q",
   "null_disagreements": 0,
   "max_abs_diff": 0.0
  },
  {
   "column": "loans_to_deposits",
   "null_disagreements": 0,
   "max_abs_diff": 0.0
  },
  {
   "column": "d4_npl_ratio",
   "null_disagreements": 0,
   "max_abs_diff": 0.0
  },
  {
   "column": "roa_mean4",
   "null_disagreements": 0,
   "max_abs_diff": 1.7763568394002505e-15
  },
  {
   "column": "npl_rank_q",
   "null_disagreements": 0,
   "max_abs_diff": 0.0
  }
 ],
 "passed": true


## Extra measurements behind design choices (results of `python run.py experiments`)

In [9]:
print(json.load(open('reports/tables/ingest_resume_test.json')))
print(json.load(open('reports/tables/layout_effects.json')))
display(pl.read_csv('reports/tables/engine_scans.csv').select('experiment','scale','setting','median_s','peak_mb'))
pl.read_csv('reports/tables/feature_signal_dev.csv').head(12)

{'quarters': 8, 'killed_after_s': 18.0, 'complete_at_kill': 3, 'files_after_kill': 3, 'partial_tmp_files_after_kill': 0, 'restart_message': '5 quarters to download (3 already complete)', 'restart_s': 12.9, 'second_restart_message': '0 quarters to download (8 already complete)', 'second_restart_s': 0.5, 'all_complete': True, 'all_sha256_match': True, 'rows': 38083}
{'rows': 1682738, 'raw_inmemory_mb': 442.7011375427246, 'typed_inmemory_mb': 435.4982805252075, 'stalp_string_mb': 3.209218978881836, 'stalp_categorical_mb': 6.419136047363281, 'repdte_string_mb': 12.838272094726562, 'repdte_date_mb': 6.419136047363281, 'sorted_file_mb': 147.5190486907959, 'sorted_row_groups': 34, 'sorted_row_groups_read_one_bank': 1, 'sorted_one_bank_query_s': 0.002859707400011757, 'unsorted_file_mb': 166.8431224822998, 'unsorted_row_groups': 34, 'unsorted_row_groups_read_one_bank': 34, 'unsorted_one_bank_query_s': 0.017949511200004053}


experiment,scale,setting,median_s,peak_mb
str,i64,str,f64,f64
"""polars_threads""",1,"""1 threads""",1.808892,498.898438
"""polars_threads""",1,"""2 threads""",1.369546,514.414062
"""polars_threads""",1,"""4 threads""",1.056234,590.46875
"""polars_threads""",3,"""1 threads""",5.98889,999.777344
"""polars_threads""",3,"""2 threads""",3.757777,1197.695312
"""polars_threads""",3,"""4 threads""",3.034976,1356.1875
"""polars_optimizer""",1,"""optimiser off""",1.062348,714.421875
"""polars_optimizer""",1,"""optimiser on""",1.058865,597.980469
"""spark_cores""",1,"""local[1]""",24.391256,1669.328125


feature,pillar,missing_pct,median_y1,median_y0,roc_auc_oriented,direction
str,str,f64,f64,f64,f64,str
"""texas_ratio""","""A""",1.254866,2.069768,0.07546,0.954482,"""higher = riskier"""
"""total_rbc_ratio""","""C""",32.479706,4.674534,15.311812,0.950882,"""lower = riskier"""
"""roa_mean4""","""dyn""",5.786708,-0.031685,0.009741,0.943595,"""lower = riskier"""
"""tier1_leverage""","""C""",5.967199,2.47313,8.839437,0.936028,"""lower = riskier"""
"""equity_to_assets""","""C""",0.233264,0.017797,0.088227,0.930944,"""lower = riskier"""
"""roa_std4""","""dyn""",5.786708,0.03547,0.003098,0.916206,"""higher = riskier"""
"""equity_rank_q""","""xs""",0.233264,0.03138,0.503544,0.911214,"""lower = riskier"""
"""npl_ratio""","""A""",1.165006,0.092473,0.009793,0.91063,"""higher = riskier"""
"""roa_q""","""E""",0.676071,-0.027078,0.009891,0.898184,"""lower = riskier"""


## Part 5 – models (results of `python run.py model`, `python run.py test` and `python run.py mllib`)

In [10]:
display(pl.read_csv('reports/tables/model_split.csv'))
display(pl.read_csv('reports/tables/model_cv_summary.csv'))
print(json.load(open('reports/tables/model_choice.json')))
print(open('reports/tables/test_log.jsonl').read())

split,from,to,rows,banks,positive_rows,failed_banks,base_rate_pct,rows_excluded_unavailable,positive_rows_excluded,failed_banks_only_in_excluded_rows
str,str,str,i64,i64,i64,i64,f64,i64,i64,i64
"""development""","""1984-03-31""","""2009-12-31""",1294242,23925,8555,2597,0.661005,1229,1229,15
"""test""","""2011-03-31""","""2025-06-30""",336835,7742,505,181,0.149925,147,147,18


model,pr_auc_mean,pr_auc_sd,roc_auc_mean,recall_top2pct_mean,brier_mean,fit_s_mean
str,f64,f64,f64,f64,f64,f64
"""hgb_all""",0.347177,0.21059,0.960305,0.709237,0.00393,11.936813
"""hgb_all_small""",0.340513,0.212252,0.962105,0.716572,0.003975,11.046674
"""hgb_all_balanced""",0.334739,0.202375,0.956337,0.701894,0.007946,13.830915
"""hgb_levels+dyn""",0.331255,0.213711,0.960856,0.705439,0.003979,11.888426
"""hgb_levels""",0.320982,0.206654,0.95775,0.700654,0.004043,8.71835
"""logit_all""",0.252076,0.218752,0.937325,0.657298,0.004535,12.62548
"""logit_all_balanced""",0.209557,0.19776,0.946319,0.647154,0.023993,19.45185
"""baseline_logit_4""",0.197679,0.207075,0.921668,0.599386,0.00517,0.974066


{'frozen_at': '2026-11-21T14:30:57Z', 'chosen_model': 'hgb_all', 'kind': 'hgb', 'feature_set': 'all', 'params': {}, 'features': ['equity_to_assets', 'tier1_leverage', 'total_rbc_ratio', 'npl_ratio', 'nonaccrual_to_assets', 'pd90_to_assets', 'ore_to_assets', 'texas_ratio', 'alll_to_loans', 'alll_coverage', 'nco_rate', 'provision_rate', 'efficiency_ratio', 'nonint_exp_to_assets', 'roa_q', 'nim_q', 'nonint_inc_to_assets', 'loans_to_deposits', 'liquid_to_assets', 'brokered_to_deposits', 'deposits_to_assets', 'log_assets', 're_loans_share', 'construction_share', 'ci_loans_share', 'consumer_share', 'asset_growth_4q', 'loan_growth_4q', 'd4_equity_to_assets', 'd4_npl_ratio', 'd1_npl_ratio', 'd4_texas_ratio', 'roa_mean4', 'roa_std4', 'npl_rank_q', 'equity_rank_q', 'roa_rank_q', 'state_npl_median', 'state_d4_npl_median', 'bank_age_years'], 'threshold': 0.023815729048690146, 'baseline_threshold': 0.019824458530090776, 'cfn_per_row_k': 584.1997256377557, 'cfp_k': 50.0, 'dev_end': '2009-12-31', 'te

In [11]:
display(pl.read_csv('reports/tables/model_test_results.csv'))
display(pl.read_csv('reports/tables/model_test_detection.csv'))
pl.read_csv('reports/tables/mllib_vs_sklearn.csv')

model,chosen,rows,positives,base_rate_pct,pr_auc,…,fp,fn,recall,precision,cost_k,cost_no_model_k
str,bool,i64,i64,f64,f64,…,i64,i64,f64,f64,f64,f64
"""hgb_all""",true,336835,505,0.149925,0.427227,…,2120,66,0.869307,0.171551,144557.181892,295020.861447
"""baseline_logit_4""",false,336835,505,0.149925,0.357118,…,2399,93,0.815842,0.146567,174280.574484,295020.861447


model,failed_banks,detected_at_threshold,detected_top2pct,median_lead_quarters
str,i64,i64,i64,f64
"""hgb_all""",181,170,172,3.220153
"""baseline_logit_4""",181,166,169,3.2092


engine,model,fit_s,predict_s,pr_auc,roc_auc
str,str,f64,f64,f64,f64
"""spark-mllib""","""logistic""",16.327601,1.15072,0.492972,0.970764
"""spark-mllib""","""gbt""",48.938064,0.791375,0.503812,0.969107
"""scikit-learn""","""logistic""",4.409057,0.082146,0.507559,0.972292
"""scikit-learn""","""gbt""",2.249814,0.115306,0.525538,0.976608
